# CatBoost：类别特征的最佳拍档
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：07_集成学习 | 源文件：CatBoost.py | 核心功能：Ordered Target Encoding、对称树、原生类别特征

## 概述

CatBoost 是 Yandex 开发的梯度提升框架，最大特点是**原生处理类别特征**——不需要手动编码。它用 Ordered Target Encoding 替代传统 Target Encoding，避免了数据泄露。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
import pandas as pd

try:
    from catboost import CatBoostClassifier, CatBoostRegressor, Pool
    HAS_CB = True
except ImportError:
    HAS_CB = False
# --- 输出结果 ---
    print("[SKIP] CatBoost 未安装，跳过本示例")
    import sys; sys.exit(0)
    HAS_CB = False
    print("CatBoost 未安装,请运行: pip install catboost\n")


## 数学原理

### 1. CatBoost 的核心创新

CatBoost（Categorical Boosting）由 Yandex 提出，两大核心改进：
- **Ordered Target Statistics**：原生处理类别特征，避免目标泄漏
- **Ordered Boosting**：解决梯度提升中的预测偏移（prediction shift）

### 2. 类别特征的 Target Statistics

传统方法用目标均值编码类别特征：

$$\hat{x}_i^k = \frac{\sum_{j=1}^{n} [x_j^k = x_i^k] \cdot y_j}{\sum_{j=1}^{n} [x_j^k = x_i^k]}$$

这会导致**目标泄漏**：$y_i$ 参与了 $\hat{x}_i^k$ 的计算。

CatBoost 的解决方案（加先验平滑）：

$$\hat{x}_i^k = \frac{\sum_{j \in D_i} [x_j^k = x_i^k] \cdot y_j + a \cdot p}{\sum_{j \in D_i} [x_j^k = x_i^k] + a}$$

其中 $D_i = \{j : \sigma(j) < \sigma(i)\}$ 是随机排列中排在 $i$ 之前的样本，$p$ 是先验值，$a$ 是平滑系数。

### 3. Ordered Boosting

标准 Gradient Boosting 的预测偏移问题：

- 训练时用 $F_{m-1}$ 计算梯度 $g_i$
- 但 $F_{m-1}$ 是在包含 $x_i$ 的数据上训练的
- 导致梯度估计有偏

CatBoost 的解决：对每个样本 $x_i$，只用在**随机排列中排在 $i$ 之前**的样本来计算梯度：

$$g_i^{(m)} = -\frac{\partial L(y_i, F^{(m-1),\sigma(i)}(x_i))}{\partial F^{(m-1),\sigma(i)}(x_i)}$$

其中 $F^{(m-1),\sigma(i)}$ 只在 $\{j : \sigma(j) < \sigma(i)\}$ 上训练。

### 4. 对称决策树（Oblivious Trees）

CatBoost 使用**对称决策树**：所有同一层的节点使用相同的分裂条件。

- 每层的分裂特征和阈值相同
- 叶节点数为 $2^d$（$d$ 为深度）
- 分裂决策可编码为二进制索引，推理极快

$$\text{leaf}(x) = \sum_{l=0}^{d-1} 2^l \cdot \mathbb{I}[f_{k_l}(x) > t_l]$$

### 5. 推理效率

由于对称树的结构，推理时：
- 每个样本沿同一条路径分裂
- 可以用位运算高效计算叶节点编号
- 缓存命中率高

### 6. 类别特征的组合

CatBoost 还会自动探索类别特征的**组合**（combinations）：

$$c_{ij} = (c_i, c_j)$$

通过贪心策略寻找最优组合，自动发现特征交互。

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `CatBoostClassifier(iterations=100)` | $M=100$ 轮 Ordered Boosting |
| `depth=3` | 对称树深度 $d=3$，叶节点数 $2^3=8$ |
| `learning_rate=0.1` | 步长收缩 $\nu$ |
| `cat_features=["城市", "性别"]` | 类别特征，用 Ordered Target Statistics 编码 |
| `early_stopping_rounds=50` | 验证集 50 轮无提升则停止 |
| `best_iteration_` | 最优迭代次数 |
| `Pool` | CatBoost 的数据容器，存储类别特征元信息 |

### 1. 分类任务

在分类任务上训练模型并评估性能。

In [ ]:
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. 基础 CatBoost

运行 2. 基础 CatBoost 的代码，观察算法在该环节的行为。

In [ ]:
cb_clf = CatBoostClassifier(
    iterations=100,
    depth=3,
    learning_rate=0.1,
    random_seed=42,
# --- 赋值/配置 ---
    verbose=0,
)
cb_clf.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

print("=== CatBoost 分类 ===")
print(f"训练准确率: {cb_clf.score(X_train, y_train):.4f}")
print(f"测试准确率: {cb_clf.score(X_test, y_test):.4f}")
print(f"最优迭代次数: {cb_clf.best_iteration_}")

### 3. 特征重要性

分析各特征对模型预测的贡献度。

In [ ]:
print("\n=== 特征重要性 ===")
importances = cb_clf.feature_importances_
for i in np.argsort(importances)[::-1]:
    print(f"  特征{i}: {importances[i]:.4f}")

### 4. 类别特征支持

运行 4. 类别特征支持 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 类别特征处理 ===")
# 构造含类别特征的数据
np.random.seed(42)
df = pd.DataFrame({
    "数值1": np.random.randn(200),
    "数值2": np.random.randn(200),
    "城市": np.random.choice(["北京", "上海", "广州"], 200),
# --- 数值计算 ---
    "性别": np.random.choice(["男", "女"], 200),
})
y_cat = (df["数值1"] + df["数值2"] > 0).astype(int)

cat_features = ["城市", "性别"]
cb_cat = CatBoostClassifier(iterations=100, depth=4, verbose=0, random_seed=42)
cb_cat.fit(df, y_cat, cat_features=cat_features)
print(f"含类别特征的准确率: {cb_cat.score(df, y_cat):.4f}")

### 5. 有序目标编码

运行 5. 有序目标编码 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== CatBoost 的 Ordered Boosting ===")
print("CatBoost 的核心创新:")
print("1. Ordered Boosting: 训练第 t 棵树时只用时间上在它之前的数据预测残差")
print("   避免 target leakage（目标泄露）")
print("2. Ordered TS: 类别特征的目标编码也是按时间顺序计算的")
# --- 输出结果 ---
print("   而非用全部数据的全局均值（避免过拟合）")

### 6. 回归任务

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== CatBoost 回归 ===")
X_r, y_r = make_regression(n_samples=500, n_features=10, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
cb_reg = CatBoostRegressor(iterations=100, depth=3, verbose=0, random_seed=42)
cb_reg.fit(X_tr, y_tr, eval_set=(X_te, y_te), early_stopping_rounds=50)
# --- 输出结果 ---
print(f"R²: {cb_reg.score(X_te, y_te):.4f}")

### 7. 不同 depth 对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== depth 对比 ===")
for d in [2, 3, 4, 5, 6, 8]:
    m = CatBoostClassifier(iterations=100, depth=d, verbose=0, random_seed=42)
    m.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    print(f"  depth={d}: 测试准确率={m.score(X_test, y_test):.4f}")

print("\n=== CatBoost 要点 ===")
print("- 原生类别特征支持（无需手动编码）")
print("- Ordered Boosting: 防止 target leakage,抗过拟合能力强")
print("- 默认参数通常就能取得很好的效果（调参需求低）")
print("- 对缺失值和类别特征的处理是内置的")
# --- 输出结果 ---
print("- 训练速度介于 XGBoost 和 LightGBM 之间")
print("- 适合：含大量类别特征的结构化数据（如推荐、广告）")

## 关键代码解释

```python
from catboost import CatBoostClassifier
model = CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1,
                           cat_features=[0, 1, 2], verbose=50)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=10)
```

## 注意事项

1. **类别特征索引**：需要指定哪些列是类别特征
2. **对称树**：CatBoost 使用对称决策树，推理更快
3. **默认参数通常就够好**：调参需求比 XGBoost/LightGBM 少

## 总结与延伸

以上代码展示了 **CatBoost** 的完整流程。

**解读要点**：
- 对比 **单模型 vs 集成模型** 的性能提升
- 关注 **特征重要性** 排名
- 观察 OOB 分数（随机森林）或 early stopping 轮次（XGBoost）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Ordered Boosting**：减少梯度估计的偏差
- **CatBoost for ranking**：排序任务的内置支持
- **三者对比**：XGBoost（稳定）/ LightGBM（快）/ CatBoost（类别特征好）
